In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
import requests
resp = requests.get('https://httpbin.org/get', timeout=10)
print(resp.status_code)

200


In [3]:
url = 'https://github.com/DataTalksClub/faq/archive/refs/heads/main.zip'
resp = requests.get(url, timeout=30)
print(resp.status_code)

200


In [4]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [5]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
print(dtc_faq[0])

{'description': 'Review and process open FAQ PRs', 'content': 'Go through all open pull requests one by one. For each PR:\n\n## 1. Show Details\n- PR number and title\n- Course and section (from PR body)\n- Related issue number\n- **ALWAYS check sort_order**: List files in target section (`ls _questions/<course>/<section>/`) to find highest number, verify PR uses next sequential\n- Full diff (use `gh pr diff <number>`)\n\n## 2. Check Against These Rules\n\n### Section Placement\n- **Kestra questions** → must be in `module-2` (workflow orchestration), NOT `general` or `module-1`\n- **Docker + Kestra** → still `module-2` (Kestra is primary topic)\n- **Docker-only** (pgAdmin, Postgres, etc.) → `module-1`\n\n### Relevance (Close If)\n- Basic Linux/SQL tutorials (vim installation, SQL JOIN concepts, etc.)\n- Generic programming not tied to course content\n- Already exists in DE zoomcamp when proposed for ML zoomcamp\n\n### Duplicates (Check Before Creating)\n- Search existing FAQs: `grep -r

In [6]:
evidently_docs = read_repo_data('evidentlyai', 'docs')

In [7]:
print(f"Evidently docs: {len(evidently_docs)}")

Evidently docs: 95


In [8]:
doc = evidently_docs[45]
print(doc.keys())
print(f"Content length: {len(doc['content'])} characters")

dict_keys(['title', 'description', 'content', 'filename'])
Content length: 21712 characters


In [10]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [11]:
test = sliding_window(evidently_docs[45]['content'], 2000, 1000)
print(f"Number of chunks: {len(test)}")
print(f"First chunk start: {test[0]['start']}")
print(f"First chunk length: {len(test[0]['chunk'])}")

Number of chunks: 21
First chunk start: 0
First chunk length: 2000


In [12]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [12]:
# text = evidently_docs[45]['content']
# sections = split_markdown_by_level(text, level=2)
# print(f"Number of sections: {len(sections)}")
# print(sections[0])

In [13]:
# evidently_chunks = []

# for doc in evidently_docs:
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')
#     sections = split_markdown_by_level(doc_content, level=2)
#     for section in sections:
#         section_doc = doc_copy.copy()
#         section_doc['section'] = section
#         evidently_chunks.append(section_doc)

In [14]:
# print(f"Total chunks: {len(evidently_chunks)}")
# print(evidently_chunks[0].keys())

In [15]:
# from groq import Groq
# from dotenv import load_dotenv
# import os

# load_dotenv('../.env')

# groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# def llm(prompt, model='llama-3.3-70b-versatile'):
#     messages = [
#         {"role": "user", "content": prompt}
#     ]
    
#     response = groq_client.chat.completions.create(
#         model=model,
#         messages=messages
#     )
    
#     return response.choices[0].message.content

In [16]:
# print(llm("Say hello in one sentence."))

In [15]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

IMPORTANT: You MUST separate each section with --- on its own line. Do not skip this.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [16]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [19]:
# test_sections = intelligent_chunking(evidently_docs[45]['content'])
# print(f"Number of sections: {len(test_sections)}")
# print(test_sections[0])

In [20]:
# from tqdm.auto import tqdm

# evidently_chunks_llm = []

# for doc in tqdm(evidently_docs[:5]):  # just first 5 docs
#     doc_copy = doc.copy()
#     doc_content = doc_copy.pop('content')
#     sections = intelligent_chunking(doc_content)
#     for section in sections:
#         section_doc = doc_copy.copy()
#         section_doc['section'] = section
#         evidently_chunks_llm.append(section_doc)

# print(f"Total chunks: {len(evidently_chunks_llm)}")

In [17]:
evidently_docs = read_repo_data('evidentlyai', 'docs')

evidently_chunks = []

for doc in evidently_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)

In [22]:
from minsearch import Index

print(len(evidently_chunks))
print(evidently_chunks[0].keys())

576
dict_keys(['start', 'chunk', 'title', 'description', 'filename'])


In [23]:
print(len(evidently_chunks))
print(evidently_chunks[0].keys())

576
dict_keys(['start', 'chunk', 'title', 'description', 'filename'])


In [24]:
from minsearch import Index

index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

index.fit(evidently_chunks)

In [25]:
query = 'What should be in a test dataset for AI evaluation?'
results = index.search(query)
print(f"Number of results: {len(results)}")
print(results[0])

Number of results: 10
{'start': 0, 'chunk': 'Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.\n\nInstead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.\n\n## Create a RAG test dataset\n\nYou can generate ground truth RAG dataset from your data source.\n\n### 1. Create a Project\n\nIn the Evidently UI, start a new Project or open an existing one.\n\n* Navigate to “Datasets” in the left menu.\n* Click “Generate” and select the “RAG” option.\n\n![](/images/synthetic/synthetic_data_select_method.png)\n\n### 2. Upload your knowledge base\n\nSelect a file containing the information your AI system retrieves from. Supported formats: Markdown (.md), CSV, TXT, PDFs. Choose how many inputs to generate.\n\n![](/images/synthetic/synthe

In [26]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
de_dtc_faq = [d for d in dtc_faq if 'data-engineering' in d['filename']]

In [27]:
faq_index = Index(
    text_fields=['question','content'],  # what fields does FAQ have?
    keyword_fields=[]
)
faq_index.fit(de_dtc_faq)

In [28]:
results = faq_index.search('Can I join the course now?')
print(f"Results: {len(results)}")
print(results[0])

Results: 10
{'id': '3f1424af17', 'question': 'Course: Can I still join the course after the start date?', 'sort_order': 3, 'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'}


In [29]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [30]:
record = de_dtc_faq[2]
text = record['question'] + ' ' + record['content']
v_doc = embedding_model.encode(text)

query = 'I just found out about the course. Can I enroll now?'
v_query = embedding_model.encode(query)

similarity = v_query.dot(v_doc)
print(f"Vector size: {v_doc.shape}")
print(f"Similarity: {similarity}")

Vector size: (768,)
Similarity: 0.5190930962562561


In [31]:
print(record)

{'id': '3f1424af17', 'question': 'Course: Can I still join the course after the start date?', 'sort_order': 3, 'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'}


In [32]:
from minsearch import VectorSearch
from tqdm.auto import tqdm
import numpy as np

faq_embeddings = []

for record in de_dtc_faq:
    text = record['question'] + ' ' + record['content']
    v_doc = embedding_model.encode(text)
    faq_embeddings.append(v_doc)

faq_embeddings = np.array(faq_embeddings)

faq_vindex = VectorSearch()
faq_vindex.fit(faq_embeddings, de_dtc_faq)

In [33]:
query = 'Can I join the course now?'
q = embedding_model.encode(query)
results = faq_vindex.search(q)
print(f"Results: {len(results)}")
print(results[0])

Results: 10
{'id': '3f1424af17', 'question': 'Course: Can I still join the course after the start date?', 'sort_order': 3, 'content': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'filename': 'faq-main/_questions/data-engineering-zoomcamp/general/003_3f1424af17_course-can-i-still-join-the-course-after-the-start.md'}


In [34]:
query = 'I just discovered the program, can I still enroll?'

# text search
text_results = faq_index.search(query)
print("TEXT SEARCH:")
print(text_results[0]['question'])

# vector search
q = embedding_model.encode(query)
vector_results = faq_vindex.search(q)
print("\nVECTOR SEARCH:")
print(vector_results[0]['question'])

TEXT SEARCH:
Course: Can I still join the course after the start date?

VECTOR SEARCH:
Course: Can I still join the course after the start date?


In [35]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Search the course FAQ database for relevant information.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results from the FAQ index.
    """
    return faq_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return faq_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results


In [36]:
query = 'Can I join the course now?'

print(f"Text search: {len(text_search(query))} results")
print(f"Vector search: {len(vector_search(query))} results")
print(f"Hybrid search: {len(hybrid_search(query))} results")

Text search: 5 results
Vector search: 5 results
Hybrid search: 6 results


In [37]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv('../.env')
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [38]:
text_search_tool = {
    "type": "function",
    "function": {
        "name": "text_search",
        "description": "Search the FAQ database",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}


In [39]:
system_prompt = """
You are a helpful assistant for a course. 
"""

question = "I just discovered the course, can I join now?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

In [40]:
import json

# Step 1: Initial call
response = groq_client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=chat_messages,
    tools=[text_search_tool],
    tool_choice='auto',
    temperature=0.5
)

# Step 2: Get the tool call
response_message = response.choices[0].message
print(response_message.tool_calls)

# Step 3: Execute the tool
tool_call = response_message.tool_calls[0]
function_args = json.loads(tool_call.function.arguments)
result = text_search(**function_args)

# Step 4: Append assistant message + tool result
chat_messages.append(response_message)
chat_messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "name": tool_call.function.name,
    "content": json.dumps(result)
})

# Step 5: Get final response
final_response = groq_client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=chat_messages,
    temperature=0.5
)

print(final_response.choices[0].message.content)

[ChatCompletionMessageToolCall(id='t7b9gw3kw', function=Function(arguments='{"query":"late registration policy"}', name='text_search'), type='function')]
You can join the course now. You don't need a confirmation email to start learning and submitting homework. Registration was just to gauge interest before the start date, and you're accepted. However, please note that if you're joining late, you can still receive the certificate as long as you complete the peer-reviewed capstone projects on time, but you don't need to do the homeworks if you join late.


In [41]:
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

model = GroqModel('llama-3.1-8b-instant')

system_prompt = """
You are a helpful assistant for a course. 
Use the search tool to find relevant information before answering questions.
If the first search doesn't give enough information, try different search terms.
"""

agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=[text_search]
)

In [42]:
question = "I just discovered the course, can I join now?"
result = await agent.run(user_prompt=question)
print(result.output)

You can still join the course after the start date. However, be aware that there will be deadlines for turning in homeworks and the final projects. So, don't leave everything for the last minute. Additionally, it's recommended to have basic coding experience, familiarity with SQL, and experience with Python (helpful but not required) to get the most out of this course.


In [43]:
for message in result.new_messages():
    print(message)
    print("---")

ModelRequest(parts=[SystemPromptPart(content="\nYou are a helpful assistant for a course. \nUse the search tool to find relevant information before answering questions.\nIf the first search doesn't give enough information, try different search terms.\n", timestamp=datetime.datetime(2026, 3, 29, 17, 8, 54, 373179, tzinfo=datetime.timezone.utc)), UserPromptPart(content='I just discovered the course, can I join now?', timestamp=datetime.datetime(2026, 3, 29, 17, 8, 54, 373189, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 29, 17, 8, 54, 373822, tzinfo=datetime.timezone.utc), run_id='a27321c8-8dbb-4ccd-8d4b-7ad82d1e874f')
---
ModelResponse(parts=[ToolCallPart(tool_name='text_search', args='{"query":"joining the course deadline"}', tool_call_id='3zd5p2fxn'), ToolCallPart(tool_name='text_search', args='{"query":"course registration deadlines"}', tool_call_id='0e2gd91av')], usage=RequestUsage(input_tokens=392, output_tokens=34), model_name='llama-3.1-8b-instant', times

In [44]:
question = "can I join late and get a certificate?"
result = await agent.run(user_prompt=question)
print(result.output)

Yes, you can join late and get a certificate. As long as you complete the peer-reviewed capstone projects on time, you can receive the certificate. You do not need to do the homeworks if you join late.


In [66]:
from pydantic_ai.messages import ModelMessagesTypeAdapter


def log_entry(agent, messages, source="user"):
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    # extract system prompt from first message if agent._instructions is empty
    system_prompt = agent._instructions
    if not system_prompt:
        try:
            system_prompt = dict_messages[0]['parts'][0]['content']
        except:
            system_prompt = ""

    return {
        "agent_name": agent.name,
        "system_prompt": system_prompt,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source
    }

In [67]:
import json
import secrets
from pathlib import Path
from datetime import datetime


LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source='user'):
    entry = log_entry(agent, messages, source)

    ts = entry['messages'][-1]['timestamp']
    # ts_obj = datetime.fromisoformat(ts.replace("Z", "+00:00"))
    ts_str = ts.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath


In [47]:
question = input()
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())


 How long does the course take ?


The course starts on January 12th, 2026, and it appears that the course duration is not explicitly stated in the search results. However, it is mentioned that the course will keep all the materials available after it finishes, so students can follow the course at their own pace after it finishes.


PosixPath('logs/agent_20260329_170926_1d77ff.json')

In [68]:
import os
for f in LOG_DIR.iterdir():
    print(f)

logs/agent_20260329_170945_ec8849.json
logs/agent_20260329_170926_1d77ff.json
logs/agent_20260329_171020_994ba9.json
logs/agent_20260329_171037_215308.json
logs/agent_20260328_001353_df78ae.json
logs/faq_agent_v2_20260329_171554_249452.json
logs/agent_20260329_170927_a86ac4.json
logs/agent_20260328_001351_e4eee3.json
logs/agent_20260328_000904_b92113.json
logs/agent_20260329_171003_836008.json
logs/agent_20260328_001352_7f1446.json
logs/agent_20260329_171055_fe23e2.json


In [69]:
def load_log_file(log_file):
    with open(log_file, 'r') as f_in:
        log_data = json.load(f_in)
        log_data['log_file'] = log_file
        return log_data

log_record = load_log_file(list(LOG_DIR.iterdir())[-1])
print(json.dumps(log_record, indent=2, default=str)[:500])

{
  "agent_name": "agent",
  "system_prompt": "\nYou are a helpful assistant for a course. \nUse the search tool to find relevant information before answering questions.\nIf the first search doesn't give enough information, try different search terms.\n",
  "provider": "groq",
  "model": "llama-3.1-8b-instant",
  "tools": [
    "text_search"
  ],
  "messages": [
    {
      "parts": [
        {
          "content": "\nYou are a helpful assistant for a course. \nUse the search tool to find releva


In [53]:
questions = [
    "can I join late and get a certificate?",
    "what do I need to do for the certificate?",
    "how do I install Kafka in Python?"
]

for q in questions:
    print(q)
    result = await agent.run(user_prompt=q)
    print(result.output)
    log_interaction_to_file(agent, result.new_messages())
    print()

can I join late and get a certificate?
Based on the search results, the answer to your question is that you can join late, but you still need to complete the course with a "live" cohort and peer-review capstone projects on time to receive a certificate. You do not need to do the homework if you join late. If you have any further questions or need clarification, feel free to ask!

what do I need to do for the certificate?
To get the certificate for the course, you need to:

1. Join the course with a "live" cohort, as certificates are not awarded for the self-paced mode.
2. Complete the peer-reviewed capstone projects on time.
3. Have your full name displayed correctly on the Certificate, which you can check and edit on the Course Management webpage.
4. Follow the instructions in the [certificates.md](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/certificates.md) file to generate the Certificate document yourself after the grading is completed.

Note that you don't

In [54]:
model = GroqModel('llama-3.1-8b-instant')

system_prompt = """
You are a helpful assistant for a course.  

Use the search tool to find relevant information from the course materials before answering questions.  

If you can find specific information through search, use it to provide accurate answers.

Always include references by citing the filename of the source material you used.  
When citing the reference, replace "faq-main" by the full path to the GitHub repository: "https://github.com/DataTalksClub/faq/blob/main/"
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.  
""".strip()

agent = Agent(
    name="faq_agent_v2",
    model=model,
    system_prompt=system_prompt,
    tools=[text_search]
)

In [55]:
question = input()
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())


 Can i join late and get a certificate ?


According to the course FAQ, you can join late and get a certificate only if you complete the peer-reviewed capstone projects on time. You do not need to do the homeworks if you join late. However, please note that late submissions of homework are not allowed, but you can still submit the homework if the form is still open after the due date.

For more information, please refer to the course FAQ in [Certificates after late joining](https://github.com/DataTalksClub/faq/blob/main/_questions/data-engineering-zoomcamp/general/014_3774a79c13_certificate-do-i-need-to-do-the-homeworks-to-get-t.md).


PosixPath('logs/faq_agent_v2_20260329_171554_249452.json')

In [56]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met. 

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do  
- answer_relevant: The response directly addresses the user's question  
- answer_clear: The answer is clear and correct  
- answer_citations: The response includes proper citations or sources when required  
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked? 

Output true/false for each check and provide a short explanation for your judgment.
""".strip()

In [57]:
from pydantic import BaseModel

class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


In [59]:
eval_agent = Agent(
    name='eval_agent',
    model= GroqModel('llama-3.3-70b-versatile'),
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist
)

In [60]:
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [70]:
log_record = load_log_file(list(LOG_DIR.iterdir())[-1])
# print(json.dumps(log_record, indent=2, default=str)[:500])

# log_record = load_log_file(log_files[-1])

instructions = log_record['system_prompt']
messages = log_record['messages']
question = messages[0]['parts'][0]['content']
answer = messages[-1]['parts'][0]['content']
log = json.dumps(messages)

user_prompt = user_prompt_format.format(
    instructions=instructions,
    question=question,
    answer=answer,
    log=log
)

result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
checklist = result.output

print(checklist.summary)
print()
for check in checklist.checklist:
    status = "✅" if check.check_pass else "❌"
    print(f"{status} {check.check_name}: {check.justification}")

The agent provided a clear and relevant answer to the user's question about installing Kafka in Python, following the instructions and using the search tool to find relevant information.

✅ instructions_follow: The agent used the search tool to find relevant information before answering questions.
✅ instructions_avoid: The agent did not do anything it was told not to do.
✅ answer_relevant: The response directly addresses the user's question about installing Kafka in Python.
✅ answer_clear: The answer is clear and provides step-by-step instructions for installing Kafka in Python.
✅ answer_citations: The response includes proper citations or sources, such as the GitHub link for installing kafka-python-ng.
✅ completeness: The response covers all key aspects of the request, including installation instructions and troubleshooting steps.
✅ tool_call_search: The search tool was invoked to find relevant information.


In [71]:
def simplify_log_messages(messages):
    log_simplified = []

    for m in messages:
        parts = []
    
        for original_part in m['parts']:
            part = original_part.copy()
            kind = part['part_kind']
    
            if kind == 'user-prompt':
                del part['timestamp']
            if kind == 'tool-call':
                del part['tool_call_id']
            if kind == 'tool-return':
                del part['tool_call_id']
                del part['metadata']
                del part['timestamp']
                # Replace actual search results with placeholder to save tokens
                part['content'] = 'RETURN_RESULTS_REDACTED'
            if kind == 'text':
                del part['id']
    
            parts.append(part)
    
        message = {
            'kind': m['kind'],
            'parts': parts
        }
    
        log_simplified.append(message)
    return log_simplified


In [73]:
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record['messages']

    instructions = log_record['system_prompt']
    question = messages[0]['parts'][0]['content']
    answer = messages[-1]['parts'][0]['content']

    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )

    result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
    return result.output 


log_record = load_log_file(list(LOG_DIR.iterdir())[0])
eval1 = await evaluate_log_record(eval_agent, log_record)


In [77]:
questions = [
    "how do I use docker on windows?",
    "can I join late and get a certificate?",
    "what do I need to do for the certificate?",
    "how do I install Kafka in Python?",
    "what are the deadlines for homework?"
]

for q in tqdm(questions):
    print(q)
    result = await agent.run(user_prompt=q)
    print(result.output)
    log_interaction_to_file(agent, result.new_messages())
    print()

  0%|          | 0/5 [00:00<?, ?it/s]

how do I use docker on windows?
To use Docker on Windows, you have two main options depending on the version of Windows you are running.

**Windows 10 Pro / 11 Pro Users**

1. Ensure Hyper-V is enabled, as Docker can use it as a backend.
2. Follow the [Enable Hyper-V Option on Windows 10 / 11](https://www.c-sharpcorner.com/article/install-and-configured-docker-desktop-in-windows-10/) tutorial.

**Windows 10 Home / 11 Home Users**

1. The 'Home' version doesn't support Hyper-V, so use WSL2 (Windows Subsystem for Linux).
2. Refer to [install WSL on Windows 11](https://pureinfotech.com/install-wsl-windows-11/) for detailed instructions.

If you encounter the \"WslRegisterDistribution failed with error: 0x800701bc\" error:

* Update the WSL2 Linux Kernel by following the guidelines at [GitHub: WSL Issue 5393](https://github.com/microsoft/WSL/issues/5393).

If you get an error during connect, ensure you are running the latest version of Docker for Windows. Download the updated version from 

In [78]:
eval_set_v2 = []

for log_file in LOG_DIR.iterdir():
    log_record = load_log_file(log_file)
    if log_record['agent_name'] != 'faq_agent_v2':
        continue
    eval_set_v2.append(log_record)

print(f"Found {len(eval_set_v2)} v2 logs")

Found 6 v2 logs


In [79]:
from tqdm.auto import tqdm

eval_results = []

for log_file in tqdm(list(LOG_DIR.iterdir())):
    log_record = load_log_file(log_file)
    if log_record['agent_name'] != 'faq_agent_v2':
        continue
    eval_result = await evaluate_log_record(eval_agent, log_record)
    eval_results.append((log_record, eval_result))

print(f"Evaluated {len(eval_results)} logs")

  0%|          | 0/17 [00:00<?, ?it/s]

Evaluated 6 logs


In [80]:
import pandas as pd

rows = []
for log_record, eval_result in eval_results:
    messages = log_record['messages']
    row = {
        'question': messages[0]['parts'][0]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }
    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)
    rows.append(row)

df_evals = pd.DataFrame(rows)
print(df_evals.mean(numeric_only=True))

instructions_follow    0.833333
instructions_avoid     1.000000
answer_relevant        1.000000
answer_clear           1.000000
answer_citations       0.666667
completeness           0.833333
tool_call_search       1.000000
dtype: float64


In [81]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about a data engineering course.

Based on the provided FAQ content, generate realistic questions that students might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model= GroqModel('llama-3.3-70b-versatile'),
    output_type=QuestionsList
)


In [82]:
import random

sample = random.sample(de_dtc_faq, 10)
prompt_docs = [d['content'] for d in sample]
prompt = json.dumps(prompt_docs)

result = await question_generator.run(prompt)
questions = result.output.questions

Task was destroyed but it is pending!
task: <Task pending name='Task-737' coro=<_async_in_context.<locals>.run_in_context() done, defined at /workspaces/AI_Agent_Crash_Course/course/.venv/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-738' coro=<Kernel.shell_main() running at /workspaces/AI_Agent_Crash_Course/course/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /workspaces/AI_Agent_Crash_Course/course/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/usr/local/lib/python3.11/codeop.py:125: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  codeob = compile(source, filename, symbol, flags, True)
Task was destroyed but it is pending!
task: <Task pending name='Task-738' coro=<Kernel.shell_main() running at /workspaces/AI_Agent_Crash_Course/course/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task

In [83]:
print(f"Generated {len(questions)} questions")
for q in questions:
    print(q)

Generated 10 questions
What steps can be taken to resolve the issue of the ny_taxi_postgres_data folder appearing empty in a Docker setup on a Windows system?
What are the possible solutions for resolving the psycopg error that occurs when the PostgreSQL client library cannot be found?
What platforms and environments are available for use in the data engineering course?
How can data be uploaded from Google Cloud Storage to BigQuery to avoid errors?
What steps can be taken to troubleshoot issues with multiple Spark sessions active and observing the incorrect one?
How can a new VM instance with a different location be created from an existing VM image in GCP?
What is the cause of the error that occurs when using a SELECT * query without specifying table names, and how can it be resolved?
Why might the option to set up a CI Job in dbt Cloud be disabled, and what can be done to resolve this issue?
How does dbt manage dependencies between models using the ref keyword?
What is the cause of t

In [86]:
import time
from tqdm.auto import tqdm

for q in tqdm(questions):
    print(q)

    result = await agent.run(user_prompt=q)
    print(result.output)

    log_interaction_to_file(
        agent,
        result.new_messages(),
        source='ai-generated'
    )

    print()
    time.sleep(30)

  0%|          | 0/10 [00:00<?, ?it/s]

What steps can be taken to resolve the issue of the ny_taxi_postgres_data folder appearing empty in a Docker setup on a Windows system?
To resolve the issue of the ny_taxi_postgres_data folder appearing empty in a Docker setup on a Windows system, you can try the following steps:

1. Run the Docker command with the absolute path quoted in the -v parameter. For example:

`winpty docker run -it -e POSTGRES_USER="root" -e POSTGRES_PASSWORD="root" -e POSTGRES_DB="ny_taxi" -v "C:\Users\abhin\dataengg\DE_Project_git_connected\DE_OLD\week1_set_up\docker_sql/ny_taxi_postgres_data:/var/lib/postgresql/data" -p 5432:5432 postgres:13`

This should resolve the visibility issue in the VS Code ny_taxi folder.

2. If the above step doesn't work, try finishing the folder path with a forward slash /. For example:

`docker run -it -e POSTGRES_USER="root" -e POSTGRES_PASSWORD="root" -e POSTGRES_DB="ny_taxi" -v /$(pwd)/ny_taxi_postgres_data:/var/lib/postgresql/data -p 5432:5432 postgres:13`

These steps sh

In [87]:
eval_set = []

for log_file in LOG_DIR.glob('*.json'):
    if 'faq_agent_v2' not in log_file.name:
        continue

    log_record = load_log_file(log_file)
    if log_record['source'] != 'ai-generated':
        continue

    eval_set.append(log_record)


In [88]:
eval_results = []

for log_record in tqdm(eval_set):
    eval_result = await evaluate_log_record(eval_agent, log_record)
    eval_results.append((log_record, eval_result))


  0%|          | 0/22 [00:00<?, ?it/s]

In [89]:
rows = []

for log_record, eval_result in eval_results:
    messages = log_record['messages']

    row = {
        'file': log_record['log_file'].name,
        'question': messages[0]['parts'][0]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)

    rows.append(row)


In [90]:
import pandas as pd

df_evals = pd.DataFrame(rows)

In [91]:
df_evals.mean(numeric_only=True)

instructions_follow    1.000000
instructions_avoid     1.000000
answer_relevant        0.954545
answer_clear           0.954545
answer_citations       0.954545
completeness           0.909091
tool_call_search       1.000000
dtype: float64